# e-SNLI CoT-SFT → DPO Pipeline
이 노트북은 GPT-2 small (124M) 모델을 e-SNLI 데이터셋에 맞춰 Fine-tuning(SFT)하고, DPO(Direct Preference Optimization)까지 수행하는 전체 파이프라인입니다.

In [1]:
# 1. GPU 활성화 상태 확인
!nvidia-smi

Wed Jun  3 16:28:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# 2. Google Drive 마운트 및 작업 디렉토리 이동
from google.colab import drive
import os

drive.mount('/content/drive')
# 본인의 Google Drive 경로에 맞춰 수정 가능
%cd /content/drive/MyDrive/서강대학교/NLP_FINAL

print('현재 작업 디렉토리:', os.getcwd())
print('프로젝트 내 Python 파일:', [f for f in os.listdir('.') if f.endswith('.py')])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/서강대학교/NLP_FINAL
현재 작업 디렉토리: /content/drive/MyDrive/서강대학교/NLP_FINAL
프로젝트 내 Python 파일: ['optimizer_test.py', 'prepare_submit.py', 'sanity_check.py', 'optimizer.py', 'prepare_gsm8k.py', 'sonnet_generation.py', 'paraphrase_detection.py', 'reasoning_generation.py', 'train_dpo.py', 'train_dpo_esnli.py', 'reasoning_generation_esnli.py', 'prepare_esnli.py']


In [4]:
# 3. 필요한 의존성 패키지 설치
!pip install einops transformers trl datasets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 20.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.7 MB/s eta 0:00:00:00:0100:01


In [5]:
# 4. e-SNLI 데이터셋 로드 및 분할 전처리
# --train_size로 SFT 학습 데이터 크기 설정 (기본값: 20000, 시간 여유가 있다면 더 늘려도 됩니다)
!python prepare_esnli.py --train_size 20000 --held_out_size 1000 --dev_size 1000

Loading training data from CSVs...
Reading data/e-SNLI/dataset/esnli_train_1.csv...
Loading dev data from CSV...
Reading data/e-SNLI/dataset/esnli_dev.csv...
Loaded datasets: Train=20000, Held-out=1000, Dev=1000
Writing SFT training dataset to data/esnli_small_train.txt...
Writing held-out dataset to data/esnli_small_held_out.txt...
Writing dev dataset to data/esnli_dev.jsonl...
Writing DPO source dataset to data/esnli_dpo_source.jsonl...
Finished creating e-SNLI datasets successfully!


## 1단계: CoT-SFT 학습 및 평가

In [7]:
# 5. SFT 모델 학습
# --mask_prompt 옵션을 추가하여 질문 토큰의 Loss는 마스킹하고 추론/결론 부분만 집중 학습합니다.
# 학습 에폭(epochs)은 기본 3으로 설정했으나 상황에 맞게 늘릴 수 있습니다.
!python reasoning_generation_esnli.py --use_gpu --epochs 3 --batch_size 8 --lr 1e-5 --mask_prompt

tokenizer_config.json: 100% 26.0/26.0 [00:00<00:00, 72.5kB/s]
vocab.json: 100% 1.04M/1.04M [00:00<00:00, 33.7MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 13.1MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 31.6MB/s]
Loaded e-SNLI examples: 20000
Loaded e-SNLI examples: 1000
Held-out examples: 1000
config.json: 100% 665/665 [00:00<00:00, 3.19MB/s]
model.safetensors: 100% 548M/548M [00:10<00:00, 54.7MB/s]
Loading weights: 100% 148/148 [00:00<00:00, 1432.48it/s, Materializing param=wte.weight]             
GPT2Model LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
train-0: 100% 2500/2500 [08:42<00:00,  4.79it/s]
Epoch 0: train loss :: 1.262.
Generating several output reasoning sequences...
Generated Sample:
0

Premise: A man in a green apron roasts a pig over coals.`
Hyp

In [ ]:
# 6. SFT 학습 모델 평가 (Dev 세트)
# 3에폭 기준 저장된 파일명: '2_1e-05-esnli.pt'
!python eval_esnli.py --checkpoint 2_1e-05-esnli.pt --use_gpu

Traceback (most recent call last):
  File "/content/drive/MyDrive/서강대학교/NLP_FINAL/eval_esnli.py", line 178, in <module>
    main()
  File "/content/drive/MyDrive/서강대학교/NLP_FINAL/eval_esnli.py", line 103, in main
    saved = torch.load(args.checkpoint, weights_only=False)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 1530, in load
    with _open_file_like(f, "rb") as opened_file:
         ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 795, in _open_file_like
    return _open_file(name_or_buffer, mode)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 776, in __init__
    super().__init__(open(name, mode))  # noqa: SIM115
                     ^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '2_1e-05-esnli.pt'


## 2단계: DPO 선호 데이터 생성 및 학습

In [8]:
# 7. DPO Rejected 데이터셋 생성
# SFT 모델의 예측 결과 중 틀린 샘플만 골라내어 chosen/rejected 쌍을 만듭니다.
!python generate_rejected_esnli.py --checkpoint 2_1e-05-esnli.pt --use_gpu

Traceback (most recent call last):
  File "/content/drive/MyDrive/서강대학교/NLP_FINAL/generate_rejected_esnli.py", line 108, in <module>
    main()
  File "/content/drive/MyDrive/서강대학교/NLP_FINAL/generate_rejected_esnli.py", line 39, in main
    saved = torch.load(args.checkpoint, weights_only=False)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 1530, in load
    with _open_file_like(f, "rb") as opened_file:
         ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 795, in _open_file_like
    return _open_file(name_or_buffer, mode)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 776, in __init__
    super().__init__(open(name, mode))  # noqa: SIM115
                     ^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '2_1e-05-esnli.pt'


In [9]:
# 8. DPO 학습 수행
# beta=0.1, lr=1e-6으로 선호도 최적화를 진행하며 outputs/dpo_esnli_model에 저장합니다.
!python train_dpo_esnli.py --sft_checkpoint 2_1e-05-esnli.pt --use_gpu --epochs 3 --batch_size 4 --lr 1e-6

object address  : 0x7a601ad5e920
object refcount : 3
object type     : 0xa284e0
object type name: KeyboardInterrupt
object repr     : KeyboardInterrupt()
lost sys.stderr
Traceback (most recent call last):


In [ ]:
# 9. 최종 DPO 학습 모델 평가
# Hugging Face 디렉토리 포맷으로 저장된 dpo_esnli_model을 로드하여 평가합니다.
!python eval_esnli.py --checkpoint outputs/dpo_esnli_model --use_gpu